# 04. 대전 전체 일별 물량 예측

## 현재 작업 범위

이 노트북은 하이브리드 투스테이지 모델링의 첫 작업 단위로 다음 항목만 수행한다.

1. Train/Test 데이터 로드 및 무결성 재검증
2. Stage 1 평시 모델의 입력 피처와 Target 확정
3. Train 내부 expanding-window 검증 Fold 구성
4. 평시 물량 단순 기준 모델 성능 측정
5. Stage 1 평시 ML 후보 비교 및 선택

> 2026년 Test는 구조만 확인하고 모델 선택이나 성능 비교에 사용하지 않는다. Stage 2 이벤트 효과 모델은 다음 작업 단위에서 진행한다.

## 1. 라이브러리 및 경로 설정

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error


RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "test.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 2. Train/Test 로드 및 무결성 검증

In [2]:
EXPECTED_COLUMNS = [
    "접수일자",
    "data_split",
    "접수지역",
    "접수통수",
    "요일",
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
    "event_count",
    "is_event",
    "month",
    "day",
    "weekday_월",
    "weekday_화",
    "weekday_수",
    "weekday_목",
    "weekday_금",
    "weekday_토",
    "weekday_일",
    "days_since_prev",
    "lag_1",
    "lag_5",
    "rolling_mean_5",
    "rolling_mean_20",
]

assert TRAIN_PATH.exists(), f"Train 파일이 없습니다: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"Test 파일이 없습니다: {TEST_PATH}"

train = pd.read_csv(
    TRAIN_PATH,
    encoding="utf-8-sig",
    parse_dates=["접수일자"],
)
test = pd.read_csv(
    TEST_PATH,
    encoding="utf-8-sig",
    parse_dates=["접수일자"],
)

assert train.columns.tolist() == EXPECTED_COLUMNS
assert test.columns.tolist() == EXPECTED_COLUMNS
assert train.shape == (475, 28)
assert test.shape == (122, 28)
assert train.isna().sum().sum() == 0
assert test.isna().sum().sum() == 0
assert train["접수일자"].is_unique
assert test["접수일자"].is_unique
assert train["접수일자"].is_monotonic_increasing
assert test["접수일자"].is_monotonic_increasing
assert train["data_split"].eq("train").all()
assert test["data_split"].eq("test").all()
assert train["접수일자"].max() < test["접수일자"].min()
assert set(train["접수일자"]).isdisjoint(set(test["접수일자"]))
assert train["is_event"].value_counts().sort_index().to_dict() == {
    0: 334,
    1: 141,
}
assert test["is_event"].value_counts().sort_index().to_dict() == {
    0: 96,
    1: 26,
}

dataset_summary = pd.DataFrame(
    {
        "구분": ["Train", "Test"],
        "행 수": [len(train), len(test)],
        "열 수": [len(train.columns), len(test.columns)],
        "시작일": [
            train["접수일자"].min().date().isoformat(),
            test["접수일자"].min().date().isoformat(),
        ],
        "종료일": [
            train["접수일자"].max().date().isoformat(),
            test["접수일자"].max().date().isoformat(),
        ],
        "평시": [
            int(train["is_event"].eq(0).sum()),
            int(test["is_event"].eq(0).sum()),
        ],
        "이벤트": [
            int(train["is_event"].eq(1).sum()),
            int(test["is_event"].eq(1).sum()),
        ],
        "결측치": [
            int(train.isna().sum().sum()),
            int(test.isna().sum().sum()),
        ],
    }
)

display(dataset_summary)

,구분,행 수,열 수,시작일,종료일,평시,이벤트,결측치
0,Train,475,28,2024-01-30,2025-12-31,334,141,0
1,Test,122,28,2026-01-02,2026-06-30,96,26,0


## 3. Stage 1 Target 및 피처 확정

In [3]:
TARGET_COLUMN = "접수통수"

EVENT_COLUMNS = [
    "제1기분 자동차세",
    "재산세(건축)",
    "정기분 주민세",
    "주민세(사업소분)",
    "재산세(토지)",
    "제2기분 자동차세",
    "사회보험료 통합",
]

STAGE1_FEATURES = [
    "month",
    "day",
    "weekday_월",
    "weekday_화",
    "weekday_수",
    "weekday_목",
    "weekday_금",
    "weekday_토",
    "weekday_일",
    "days_since_prev",
    "lag_1",
    "lag_5",
    "rolling_mean_5",
    "rolling_mean_20",
]

EXCLUDED_MODEL_COLUMNS = [
    "접수일자",
    "data_split",
    "접수지역",
    TARGET_COLUMN,
    "요일",
    "event_count",
    "is_event",
    *EVENT_COLUMNS,
]

assert TARGET_COLUMN in train.columns
assert set(STAGE1_FEATURES).issubset(train.columns)
assert set(EXCLUDED_MODEL_COLUMNS).issubset(train.columns)
assert set(STAGE1_FEATURES).isdisjoint(EXCLUDED_MODEL_COLUMNS)
assert train[STAGE1_FEATURES].notna().all(axis=None)
assert test[STAGE1_FEATURES].notna().all(axis=None)
assert train[TARGET_COLUMN].gt(0).all()
assert test[TARGET_COLUMN].gt(0).all()

baseline_train_all = train.loc[train["is_event"].eq(0)].copy()
event_train_all = train.loc[train["is_event"].eq(1)].copy()

feature_definition = pd.DataFrame(
    {
        "항목": [
            "Target",
            "Stage 1 피처 수",
            "Stage 1 학습 가능 평시 행",
            "Stage 2 후보 이벤트 행",
            "Test 사용 여부(현재 단계)",
        ],
        "값": [
            TARGET_COLUMN,
            len(STAGE1_FEATURES),
            len(baseline_train_all),
            len(event_train_all),
            "구조 검증만 수행",
        ],
    }
)

display(feature_definition)
display(pd.DataFrame({"Stage 1 피처": STAGE1_FEATURES}))

,항목,값
0,Target,접수통수
1,Stage 1 피처 수,14
2,Stage 1 학습 가능 평시 행,334
3,Stage 2 후보 이벤트 행,141
4,Test 사용 여부(현재 단계),구조 검증만 수행


,Stage 1 피처
0,month
1,day
2,weekday_월
3,weekday_화
4,weekday_수
5,weekday_목
6,weekday_금
7,weekday_토
8,weekday_일
9,days_since_prev


## 4. Train 내부 expanding-window Fold 구성

In [4]:
FOLD_SPECS = [
    {
        "fold": "Fold 1",
        "train_start": "2024-01-30",
        "train_end": "2024-06-30",
        "valid_start": "2024-07-01",
        "valid_end": "2024-12-31",
    },
    {
        "fold": "Fold 2",
        "train_start": "2024-01-30",
        "train_end": "2024-12-31",
        "valid_start": "2025-01-01",
        "valid_end": "2025-06-30",
    },
    {
        "fold": "Fold 3",
        "train_start": "2024-01-30",
        "train_end": "2025-06-30",
        "valid_start": "2025-07-01",
        "valid_end": "2025-12-31",
    },
]

time_folds = {}
fold_summary_rows = []

for spec in FOLD_SPECS:
    fold_name = spec["fold"]
    train_start = pd.Timestamp(spec["train_start"])
    train_end = pd.Timestamp(spec["train_end"])
    valid_start = pd.Timestamp(spec["valid_start"])
    valid_end = pd.Timestamp(spec["valid_end"])

    fold_train_all = train.loc[
        train["접수일자"].between(train_start, train_end)
    ].copy()
    fold_valid_all = train.loc[
        train["접수일자"].between(valid_start, valid_end)
    ].copy()

    fold_train_baseline = fold_train_all.loc[
        fold_train_all["is_event"].eq(0)
    ].copy()
    fold_valid_baseline = fold_valid_all.loc[
        fold_valid_all["is_event"].eq(0)
    ].copy()

    assert not fold_train_baseline.empty
    assert not fold_valid_baseline.empty
    assert fold_train_baseline["접수일자"].max() < (
        fold_valid_baseline["접수일자"].min()
    )
    assert fold_train_baseline["접수일자"].max() < test["접수일자"].min()
    assert fold_valid_baseline["접수일자"].max() < test["접수일자"].min()

    time_folds[fold_name] = {
        "train_all": fold_train_all,
        "valid_all": fold_valid_all,
        "train_baseline": fold_train_baseline,
        "valid_baseline": fold_valid_baseline,
    }

    fold_summary_rows.append(
        {
            "Fold": fold_name,
            "학습 전체": len(fold_train_all),
            "학습 평시": len(fold_train_baseline),
            "학습 이벤트": int(fold_train_all["is_event"].eq(1).sum()),
            "검증 전체": len(fold_valid_all),
            "검증 평시": len(fold_valid_baseline),
            "검증 이벤트": int(fold_valid_all["is_event"].eq(1).sum()),
            "학습 종료일": (
                fold_train_all["접수일자"].max().date().isoformat()
            ),
            "검증 시작일": (
                fold_valid_all["접수일자"].min().date().isoformat()
            ),
            "검증 종료일": (
                fold_valid_all["접수일자"].max().date().isoformat()
            ),
        }
    )

fold_summary = pd.DataFrame(fold_summary_rows)
display(fold_summary)

,Fold,학습 전체,학습 평시,학습 이벤트,검증 전체,검증 평시,검증 이벤트,학습 종료일,검증 시작일,검증 종료일
0,Fold 1,104,82,22,124,80,44,2024-06-28,2024-07-01,2024-12-31
1,Fold 2,228,162,66,123,94,29,2024-12-31,2025-01-02,2025-06-30
2,Fold 3,351,256,95,124,78,46,2025-06-30,2025-07-01,2025-12-31


## 5. 회귀 평가 지표

In [5]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    assert y_true.shape == y_pred.shape
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()
    assert (y_true > 0).all(), "MAPE 계산을 위해 실제값이 0보다 커야 합니다."

    absolute_error = np.abs(y_true - y_pred)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE(%)": np.mean(absolute_error / y_true) * 100,
        "WAPE(%)": absolute_error.sum() / np.abs(y_true).sum() * 100,
    }


metric_test = regression_metrics(
    y_true=np.array([100.0, 200.0]),
    y_pred=np.array([90.0, 220.0]),
)

assert np.isclose(metric_test["MAE"], 15.0)
assert np.isclose(metric_test["RMSE"], np.sqrt(250.0))
assert np.isclose(metric_test["MAPE(%)"], 10.0)
assert np.isclose(metric_test["WAPE(%)"], 10.0)

display(pd.DataFrame([metric_test], index=["지표 함수 자체 검증"]))

,MAE,RMSE,MAPE(%),WAPE(%)
지표 함수 자체 검증,15.000,15.811,10.000,10.000


## 6. 평시 단순 기준 모델 평가

다음 기준 모델을 Train 내부 평시 검증 구간에서 비교한다.

- 학습 평시 평균
- 학습 평시 중앙값
- 직전 관측일 물량(`lag_1`)
- 직전 5개 관측일 전 물량(`lag_5`)
- 직전 5개 관측일 평균(`rolling_mean_5`)
- 직전 20개 관측일 평균(`rolling_mean_20`)
- 학습 평시 요일별 평균

Lag·이동평균은 매일 실제 물량이 갱신되는 rolling 1-step 예측 기준이다.

In [6]:
def weekday_mean_prediction(train_frame, valid_frame):
    weekday_means = train_frame.groupby("요일")[TARGET_COLUMN].mean()
    fallback = train_frame[TARGET_COLUMN].median()
    return valid_frame["요일"].map(weekday_means).fillna(fallback).to_numpy()


naive_result_rows = []

for fold_name, fold_data in time_folds.items():
    fold_train = fold_data["train_baseline"]
    fold_valid = fold_data["valid_baseline"]
    y_valid = fold_valid[TARGET_COLUMN].to_numpy()

    predictions = {
        "train_mean": np.full(
            len(fold_valid),
            fold_train[TARGET_COLUMN].mean(),
        ),
        "train_median": np.full(
            len(fold_valid),
            fold_train[TARGET_COLUMN].median(),
        ),
        "lag_1": fold_valid["lag_1"].to_numpy(),
        "lag_5": fold_valid["lag_5"].to_numpy(),
        "rolling_mean_5": fold_valid["rolling_mean_5"].to_numpy(),
        "rolling_mean_20": fold_valid["rolling_mean_20"].to_numpy(),
        "weekday_mean": weekday_mean_prediction(
            fold_train,
            fold_valid,
        ),
    }

    for model_name, y_pred in predictions.items():
        assert len(y_pred) == len(y_valid)
        assert np.isfinite(y_pred).all()

        metrics = regression_metrics(y_valid, y_pred)
        naive_result_rows.append(
            {
                "Fold": fold_name,
                "모델": model_name,
                "검증 평시 행": len(fold_valid),
                **metrics,
            }
        )

naive_fold_results = pd.DataFrame(naive_result_rows)
naive_summary = (
    naive_fold_results.groupby("모델", as_index=False)
    .agg(
        Fold수=("Fold", "nunique"),
        평균_MAE=("MAE", "mean"),
        평균_RMSE=("RMSE", "mean"),
        평균_MAPE=("MAPE(%)", "mean"),
        평균_WAPE=("WAPE(%)", "mean"),
        MAE_표준편차=("MAE", "std"),
    )
    .sort_values(["평균_MAE", "평균_RMSE"])
    .reset_index(drop=True)
)

assert naive_fold_results.shape[0] == len(FOLD_SPECS) * 7
assert naive_summary["Fold수"].eq(3).all()

display(naive_fold_results)
display(naive_summary)

,Fold,모델,검증 평시 행,MAE,RMSE,MAPE(%),WAPE(%)
0,Fold 1,train_mean,80,"35,598.148","42,722.936",70.759,44.676
1,Fold 1,train_median,80,"33,904.900","40,461.359",61.908,42.551
2,Fold 1,lag_1,80,"45,960.875","56,626.282",73.484,57.682
3,Fold 1,lag_5,80,"52,290.238","105,019.598",85.612,65.625
4,Fold 1,rolling_mean_5,80,"43,042.355","55,563.351",78.247,54.019
5,Fold 1,rolling_mean_20,80,"33,499.026","42,210.965",65.593,42.042
6,Fold 1,weekday_mean,80,"29,963.441","38,010.063",54.290,37.605
7,Fold 2,train_mean,94,"42,253.446","64,101.991",72.012,49.781
8,Fold 2,train_median,94,"40,870.319","64,133.556",66.098,48.151
9,Fold 2,lag_1,94,"55,719.777","80,121.113",82.504,65.646


,모델,Fold수,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE,MAE_표준편차
0,weekday_mean,3,"31,872.351","45,141.790",60.405,41.929,"5,804.907"
1,train_median,3,"34,370.112","45,891.234",65.788,45.115,"6,280.537"
2,rolling_mean_20,3,"36,662.765","48,256.940",70.074,48.039,"8,598.670"
3,train_mean,3,"36,728.515","48,271.412",75.368,48.466,"5,055.433"
4,rolling_mean_5,3,"41,377.300","57,149.321",76.595,54.147,"8,239.767"
5,lag_1,3,"45,859.076","60,492.118",74.253,59.961,"9,911.992"
6,lag_5,3,"49,751.244","92,102.388",90.648,65.798,"4,882.888"


## 7. 첫 작업 단위 결과

In [7]:
best_naive_by_mae = naive_summary.iloc[0]

checkpoint_summary = pd.DataFrame(
    {
        "항목": [
            "Train/Test 무결성",
            "Stage 1 평시 Train",
            "Stage 1 피처 수",
            "Train 내부 Fold",
            "단순 기준 모델 수",
            "평균 MAE 기준 최우수 단순 모델",
            "2026년 Test 성능 확인",
        ],
        "결과": [
            "통과",
            f"{len(baseline_train_all):,}행",
            len(STAGE1_FEATURES),
            len(time_folds),
            naive_summary["모델"].nunique(),
            best_naive_by_mae["모델"],
            "수행하지 않음",
        ],
    }
)

display(checkpoint_summary)

print("첫 모델링 작업 단위 완료")
print("- 데이터 로드 및 무결성 검증: 완료")
print("- Stage 1 피처·Target 정의: 완료")
print("- Train 내부 시간순 Fold: 완료")
print("- 평시 단순 기준 모델 평가: 완료")
print("- ML 모델 및 Stage 2 학습: 수행하지 않음")
print("- 2026년 Test 모델 성능 확인: 수행하지 않음")

,항목,결과
0,Train/Test 무결성,통과
1,Stage 1 평시 Train,334행
2,Stage 1 피처 수,14
3,Train 내부 Fold,3
4,단순 기준 모델 수,7
5,평균 MAE 기준 최우수 단순 모델,weekday_mean
6,2026년 Test 성능 확인,수행하지 않음


첫 모델링 작업 단위 완료
- 데이터 로드 및 무결성 검증: 완료
- Stage 1 피처·Target 정의: 완료
- Train 내부 시간순 Fold: 완료
- 평시 단순 기준 모델 평가: 완료
- ML 모델 및 Stage 2 학습: 수행하지 않음
- 2026년 Test 모델 성능 확인: 수행하지 않음


## 8. Stage 1 평시 ML 후보 정의

추가 패키지 설치 없이 현재 scikit-learn 환경에서 다음 회귀 모델을 비교한다.

- Ridge: `alpha=1`, `alpha=10`
- Random Forest
- Extra Trees
- HistGradientBoosting

각 모델은 원본 Target과 `log1p` Target을 각각 비교한다. 모델 선택 기준은 Train 내부 3개 Fold의 평균 MAE이며, 평균 RMSE를 2차 기준으로 사용한다. 2026년 Test는 사용하지 않는다.

In [8]:
from time import perf_counter

from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def ridge_factory(alpha):
    return make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha),
    )


def random_forest_factory():
    return RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def extra_trees_factory():
    return ExtraTreesRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def hist_gradient_boosting_factory():
    return HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=10,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    )


ML_MODEL_SPECS = [
    {
        "모델": "ridge_alpha_1",
        "Target변환": "raw",
        "factory": lambda: ridge_factory(alpha=1.0),
    },
    {
        "모델": "ridge_alpha_1",
        "Target변환": "log1p",
        "factory": lambda: ridge_factory(alpha=1.0),
    },
    {
        "모델": "ridge_alpha_10",
        "Target변환": "raw",
        "factory": lambda: ridge_factory(alpha=10.0),
    },
    {
        "모델": "ridge_alpha_10",
        "Target변환": "log1p",
        "factory": lambda: ridge_factory(alpha=10.0),
    },
    {
        "모델": "random_forest",
        "Target변환": "raw",
        "factory": random_forest_factory,
    },
    {
        "모델": "random_forest",
        "Target변환": "log1p",
        "factory": random_forest_factory,
    },
    {
        "모델": "extra_trees",
        "Target변환": "raw",
        "factory": extra_trees_factory,
    },
    {
        "모델": "extra_trees",
        "Target변환": "log1p",
        "factory": extra_trees_factory,
    },
    {
        "모델": "hist_gradient_boosting",
        "Target변환": "raw",
        "factory": hist_gradient_boosting_factory,
    },
    {
        "모델": "hist_gradient_boosting",
        "Target변환": "log1p",
        "factory": hist_gradient_boosting_factory,
    },
]

assert len(ML_MODEL_SPECS) == 10

model_spec_summary = pd.DataFrame(
    [
        {
            "모델": spec["모델"],
            "Target변환": spec["Target변환"],
        }
        for spec in ML_MODEL_SPECS
    ]
)

display(model_spec_summary)

,모델,Target변환
0,ridge_alpha_1,raw
1,ridge_alpha_1,log1p
2,ridge_alpha_10,raw
3,ridge_alpha_10,log1p
4,random_forest,raw
5,random_forest,log1p
6,extra_trees,raw
7,extra_trees,log1p
8,hist_gradient_boosting,raw
9,hist_gradient_boosting,log1p


## 9. Stage 1 평시 ML Fold 비교

각 Fold에서 이벤트가 없는 평시 행만 학습·검증한다. 로그 Target 모델은 `log1p`로 학습하고 `expm1`으로 원 단위에 복원한 뒤 지표를 계산한다. 예측값을 임의로 0 이상으로 자르지 않으며 음수 예측 건수를 별도로 기록한다.

In [9]:
def fit_predict_model(spec, x_train, y_train, x_valid):
    model = spec["factory"]()
    target_transform = spec["Target변환"]

    if target_transform == "log1p":
        fit_target = np.log1p(y_train)
    elif target_transform == "raw":
        fit_target = y_train
    else:
        raise ValueError(f"지원하지 않는 Target 변환: {target_transform}")

    started_at = perf_counter()
    model.fit(x_train, fit_target)
    prediction = model.predict(x_valid)
    elapsed_seconds = perf_counter() - started_at

    if target_transform == "log1p":
        prediction = np.expm1(prediction)

    return model, np.asarray(prediction, dtype=float), elapsed_seconds


stage1_ml_result_rows = []

for fold_name, fold_data in time_folds.items():
    fold_train = fold_data["train_baseline"]
    fold_valid = fold_data["valid_baseline"]

    x_train = fold_train[STAGE1_FEATURES]
    y_train = fold_train[TARGET_COLUMN].to_numpy()
    x_valid = fold_valid[STAGE1_FEATURES]
    y_valid = fold_valid[TARGET_COLUMN].to_numpy()

    for spec in ML_MODEL_SPECS:
        _, y_pred, elapsed_seconds = fit_predict_model(
            spec=spec,
            x_train=x_train,
            y_train=y_train,
            x_valid=x_valid,
        )

        assert y_pred.shape == y_valid.shape
        assert np.isfinite(y_pred).all()

        metrics = regression_metrics(y_valid, y_pred)
        stage1_ml_result_rows.append(
            {
                "Fold": fold_name,
                "모델": spec["모델"],
                "Target변환": spec["Target변환"],
                "학습 평시 행": len(fold_train),
                "검증 평시 행": len(fold_valid),
                "음수 예측": int((y_pred < 0).sum()),
                "학습·예측 초": elapsed_seconds,
                **metrics,
            }
        )

stage1_ml_fold_results = pd.DataFrame(stage1_ml_result_rows)
stage1_ml_summary = (
    stage1_ml_fold_results.groupby(
        ["모델", "Target변환"],
        as_index=False,
    )
    .agg(
        Fold수=("Fold", "nunique"),
        평균_MAE=("MAE", "mean"),
        평균_RMSE=("RMSE", "mean"),
        평균_MAPE=("MAPE(%)", "mean"),
        평균_WAPE=("WAPE(%)", "mean"),
        MAE_표준편차=("MAE", "std"),
        음수예측_합계=("음수 예측", "sum"),
        총실행초=("학습·예측 초", "sum"),
    )
    .sort_values(["평균_MAE", "평균_RMSE"])
    .reset_index(drop=True)
)

assert stage1_ml_fold_results.shape[0] == (
    len(FOLD_SPECS) * len(ML_MODEL_SPECS)
)
assert stage1_ml_summary["Fold수"].eq(3).all()

display(stage1_ml_fold_results)
display(stage1_ml_summary)

,Fold,모델,Target변환,학습 평시 행,검증 평시 행,음수 예측,학습·예측 초,MAE,RMSE,MAPE(%),WAPE(%)
0,Fold 1,ridge_alpha_1,raw,82,80,0,0.011,"28,420.371","40,290.117",40.791,35.668
1,Fold 1,ridge_alpha_1,log1p,82,80,0,0.002,"32,879.853","44,856.944",39.783,41.265
2,Fold 1,ridge_alpha_10,raw,82,80,0,0.002,"28,134.643","39,620.220",40.661,35.309
3,Fold 1,ridge_alpha_10,log1p,82,80,0,0.002,"32,292.900","43,979.773",39.625,40.528
4,Fold 1,random_forest,raw,82,80,0,0.303,"31,525.554","41,243.934",56.419,39.565
5,Fold 1,random_forest,log1p,82,80,0,0.301,"27,529.877","37,157.440",40.947,34.550
6,Fold 1,extra_trees,raw,82,80,0,0.206,"29,736.617","38,267.334",50.201,37.320
7,Fold 1,extra_trees,log1p,82,80,0,0.199,"28,591.106","38,138.327",41.212,35.882
8,Fold 1,hist_gradient_boosting,raw,82,80,1,0.225,"36,796.992","48,247.162",63.366,46.181
9,Fold 1,hist_gradient_boosting,log1p,82,80,0,0.216,"33,333.418","43,878.511",47.203,41.834


,모델,Target변환,Fold수,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE,MAE_표준편차,음수예측_합계,총실행초
0,extra_trees,log1p,3,"28,093.709","41,342.505",45.845,36.614,"6,848.386",0,0.603
1,random_forest,log1p,3,"28,459.691","41,624.004",47.319,37.196,"6,617.417",0,0.897
2,extra_trees,raw,3,"31,046.461","42,148.588",57.645,40.637,"6,897.641",0,0.607
3,ridge_alpha_10,log1p,3,"31,205.047","47,128.756",50.152,40.859,"6,066.961",0,0.006
4,ridge_alpha_1,log1p,3,"31,389.117","47,381.075",50.126,41.105,"6,008.767",0,0.006
5,ridge_alpha_10,raw,3,"31,608.579","46,392.581",57.361,41.743,"5,657.057",0,0.006
6,ridge_alpha_1,raw,3,"31,630.056","46,938.795",57.092,41.787,"5,364.287",0,0.016
7,random_forest,raw,3,"34,155.716","46,631.835",65.653,44.984,"6,188.071",0,0.912
8,hist_gradient_boosting,log1p,3,"34,701.343","48,843.184",57.246,45.563,"6,610.733",0,1.120
9,hist_gradient_boosting,raw,3,"38,351.721","52,445.438",70.055,50.148,"8,869.537",1,1.179


## 10. Stage 1 후보 선택

In [10]:
best_naive = naive_summary.iloc[0]
best_ml = stage1_ml_summary.iloc[0]

stage1_method_comparison = pd.DataFrame(
    [
        {
            "구분": "단순 기준",
            "후보": best_naive["모델"],
            "Target변환": "해당 없음",
            "평균_MAE": best_naive["평균_MAE"],
            "평균_RMSE": best_naive["평균_RMSE"],
            "평균_MAPE": best_naive["평균_MAPE"],
            "평균_WAPE": best_naive["평균_WAPE"],
        },
        {
            "구분": "ML",
            "후보": best_ml["모델"],
            "Target변환": best_ml["Target변환"],
            "평균_MAE": best_ml["평균_MAE"],
            "평균_RMSE": best_ml["평균_RMSE"],
            "평균_MAPE": best_ml["평균_MAPE"],
            "평균_WAPE": best_ml["평균_WAPE"],
        },
    ]
).sort_values(["평균_MAE", "평균_RMSE"]).reset_index(drop=True)

selected_stage1_candidate = stage1_method_comparison.iloc[0].to_dict()
ml_mae_improvement_vs_naive = (
    (best_naive["평균_MAE"] - best_ml["평균_MAE"])
    / best_naive["평균_MAE"]
    * 100
)

display(stage1_method_comparison)

print("Stage 1 평시 ML 비교 완료")
print(
    "- 최우수 단순 기준: "
    f"{best_naive['모델']} "
    f"(평균 MAE {best_naive['평균_MAE']:,.1f})"
)
print(
    "- 최우수 ML: "
    f"{best_ml['모델']} / {best_ml['Target변환']} "
    f"(평균 MAE {best_ml['평균_MAE']:,.1f})"
)
print(
    "- 최우수 ML의 단순 기준 대비 MAE 개선율: "
    f"{ml_mae_improvement_vs_naive:,.2f}%"
)
print(
    "- Stage 1 현재 선택 후보: "
    f"{selected_stage1_candidate['구분']} / "
    f"{selected_stage1_candidate['후보']} / "
    f"{selected_stage1_candidate['Target변환']}"
)
print("- 2026년 Test 성능 확인: 수행하지 않음")
print("- Stage 2 이벤트 효과 모델: 수행하지 않음")

,구분,후보,Target변환,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE
0,ML,extra_trees,log1p,"28,093.709","41,342.505",45.845,36.614
1,단순 기준,weekday_mean,해당 없음,"31,872.351","45,141.790",60.405,41.929


Stage 1 평시 ML 비교 완료
- 최우수 단순 기준: weekday_mean (평균 MAE 31,872.4)
- 최우수 ML: extra_trees / log1p (평균 MAE 28,093.7)
- 최우수 ML의 단순 기준 대비 MAE 개선율: 11.86%
- Stage 1 현재 선택 후보: ML / extra_trees / log1p
- 2026년 Test 성능 확인: 수행하지 않음
- Stage 2 이벤트 효과 모델: 수행하지 않음


## 11. 선택된 Stage 1 모델의 시간순 OOF 예측

Stage 2의 이벤트 효과를 계산할 때 실제 미래 정보를 사용하지 않도록, 각 검증 구간보다 앞선 평시 데이터만으로 Stage 1 모델을 다시 학습한다. 이렇게 만든 OOF(Out-of-Fold) 베이스라인 예측을 전체 검증일에 적용하고, 이벤트 날짜의 `실제 물량 - 베이스라인 예측 물량`을 Stage 2 학습 Target으로 사용한다.

- OOF 생성 범위: 2024년 하반기 ~ 2025년 하반기
- Stage 1 학습 데이터: 각 Fold 검증 시작일 이전의 평시 행만 사용
- 2024년 상반기: 최초 학습 구간이므로 OOF 잔차 학습 대상에서 제외
- 2026년 Test: 이 단계에서는 성능을 확인하지 않음

In [11]:
selected_stage1_spec = next(
    spec
    for spec in ML_MODEL_SPECS
    if (
        spec["모델"] == best_ml["모델"]
        and spec["Target변환"] == best_ml["Target변환"]
    )
)

stage1_oof_frames = []

for fold_name, fold_data in time_folds.items():
    fold_train = fold_data["train_baseline"]
    fold_valid = fold_data["valid_all"]

    _, baseline_prediction_raw, _ = fit_predict_model(
        spec=selected_stage1_spec,
        x_train=fold_train[STAGE1_FEATURES],
        y_train=fold_train[TARGET_COLUMN].to_numpy(),
        x_valid=fold_valid[STAGE1_FEATURES],
    )

    fold_oof = fold_valid.copy()
    fold_oof["Fold"] = fold_name
    fold_oof["stage1_baseline_prediction_raw"] = baseline_prediction_raw
    fold_oof["stage1_baseline_prediction"] = np.clip(
        baseline_prediction_raw,
        a_min=0,
        a_max=None,
    )
    fold_oof["event_residual"] = (
        fold_oof[TARGET_COLUMN]
        - fold_oof["stage1_baseline_prediction"]
    )
    stage1_oof_frames.append(fold_oof)

stage1_oof = (
    pd.concat(stage1_oof_frames, ignore_index=True)
    .sort_values("접수일자")
    .reset_index(drop=True)
)

expected_oof_rows = sum(
    len(fold_data["valid_all"])
    for fold_data in time_folds.values()
)

assert len(stage1_oof) == expected_oof_rows
assert stage1_oof["접수일자"].is_unique
assert stage1_oof["접수일자"].is_monotonic_increasing
assert stage1_oof["접수일자"].max() < test["접수일자"].min()
assert stage1_oof["stage1_baseline_prediction"].ge(0).all()
assert stage1_oof["event_residual"].notna().all()

stage1_oof_metric_rows = []
for segment_name, segment_mask in {
    "전체": pd.Series(True, index=stage1_oof.index),
    "평시": stage1_oof["is_event"].eq(0),
    "이벤트": stage1_oof["is_event"].eq(1),
}.items():
    segment = stage1_oof.loc[segment_mask]
    stage1_oof_metric_rows.append(
        {
            "구간": segment_name,
            "행 수": len(segment),
            **regression_metrics(
                segment[TARGET_COLUMN],
                segment["stage1_baseline_prediction"],
            ),
        }
    )

stage1_oof_metrics = pd.DataFrame(stage1_oof_metric_rows)
stage1_oof_summary = pd.DataFrame(
    {
        "항목": [
            "OOF 시작일",
            "OOF 종료일",
            "OOF 전체 행",
            "OOF 이벤트 행",
            "음수 원예측",
            "2026년 Test 성능 확인",
        ],
        "값": [
            stage1_oof["접수일자"].min().date().isoformat(),
            stage1_oof["접수일자"].max().date().isoformat(),
            len(stage1_oof),
            int(stage1_oof["is_event"].eq(1).sum()),
            int(
                stage1_oof["stage1_baseline_prediction_raw"].lt(0).sum()
            ),
            "수행하지 않음",
        ],
    }
)

display(stage1_oof_summary)
display(stage1_oof_metrics)

,항목,값
0,OOF 시작일,2024-07-01
1,OOF 종료일,2025-12-31
2,OOF 전체 행,371
3,OOF 이벤트 행,119
4,음수 원예측,0
5,2026년 Test 성능 확인,수행하지 않음


,구간,행 수,MAE,RMSE,MAPE(%),WAPE(%)
0,전체,371,"37,200.249","71,194.098",412.015,44.011
1,평시,252,"28,515.823","44,635.263",46.090,37.226
2,이벤트,119,"55,590.797","107,624.821","1,186.917",54.876


## 12. Stage 2 이벤트 잔차 모델의 시간순 검증

이벤트 행만 사용해 Stage 1 OOF 잔차를 학습한다. 검증 시점보다 앞서 생성된 OOF 잔차만 학습에 사용하며, 최종 비교는 잔차 자체가 아니라 `Stage 1 베이스라인 + Stage 2 조정값`의 실제 물량 예측 오차로 수행한다.

비교 후보는 Stage 1 단독, 전체 이벤트 잔차의 평균·중앙값, 이벤트 종류별 평균·중앙값, Ridge, Random Forest, Extra Trees, HistGradientBoosting이다. 잔차에는 음수가 존재할 수 있으므로 Stage 2 Target에는 로그 변환을 적용하지 않는다.

In [12]:
STAGE2_FEATURES = [
    *EVENT_COLUMNS,
    "event_count",
    "month",
    "day",
    "weekday_월",
    "weekday_화",
    "weekday_수",
    "weekday_목",
    "weekday_금",
    "weekday_토",
    "weekday_일",
    "days_since_prev",
    "stage1_baseline_prediction",
]

stage2_event_oof = stage1_oof.loc[
    stage1_oof["is_event"].eq(1)
].copy()

STAGE2_FOLD_SPECS = [
    {
        "Fold": "Stage 2 Fold 1",
        "학습 OOF Fold": ["Fold 1"],
        "검증 OOF Fold": "Fold 2",
    },
    {
        "Fold": "Stage 2 Fold 2",
        "학습 OOF Fold": ["Fold 1", "Fold 2"],
        "검증 OOF Fold": "Fold 3",
    },
]

STAGE2_ML_SPECS = [
    {
        "후보": "ridge_alpha_1",
        "factory": lambda: ridge_factory(alpha=1.0),
    },
    {
        "후보": "ridge_alpha_10",
        "factory": lambda: ridge_factory(alpha=10.0),
    },
    {
        "후보": "random_forest",
        "factory": random_forest_factory,
    },
    {
        "후보": "extra_trees",
        "factory": extra_trees_factory,
    },
    {
        "후보": "hist_gradient_boosting",
        "factory": hist_gradient_boosting_factory,
    },
]


def event_stat_adjustment(train_frame, valid_frame, statistic):
    if statistic == "mean":
        stat_function = pd.Series.mean
    elif statistic == "median":
        stat_function = pd.Series.median
    else:
        raise ValueError(f"지원하지 않는 통계량: {statistic}")

    global_adjustment = float(
        stat_function(train_frame["event_residual"])
    )
    event_adjustments = {}
    for event_column in EVENT_COLUMNS:
        event_residuals = train_frame.loc[
            train_frame[event_column].eq(1),
            "event_residual",
        ]
        if not event_residuals.empty:
            event_adjustments[event_column] = float(
                stat_function(event_residuals)
            )

    predictions = []
    for _, row in valid_frame.iterrows():
        active_adjustments = [
            event_adjustments[event_column]
            for event_column in EVENT_COLUMNS
            if row[event_column] == 1
            and event_column in event_adjustments
        ]
        predictions.append(
            float(np.mean(active_adjustments))
            if active_adjustments
            else global_adjustment
        )

    return np.asarray(predictions, dtype=float)


stage2_result_rows = []

for fold_spec in STAGE2_FOLD_SPECS:
    stage2_train = stage2_event_oof.loc[
        stage2_event_oof["Fold"].isin(fold_spec["학습 OOF Fold"])
    ].copy()
    stage2_valid = stage2_event_oof.loc[
        stage2_event_oof["Fold"].eq(fold_spec["검증 OOF Fold"])
    ].copy()

    assert not stage2_train.empty
    assert not stage2_valid.empty
    assert stage2_train["접수일자"].max() < stage2_valid["접수일자"].min()

    candidate_adjustments = {
        "stage1_only": np.zeros(len(stage2_valid), dtype=float),
        "global_mean": np.repeat(
            stage2_train["event_residual"].mean(),
            len(stage2_valid),
        ),
        "global_median": np.repeat(
            stage2_train["event_residual"].median(),
            len(stage2_valid),
        ),
        "event_type_mean": event_stat_adjustment(
            stage2_train,
            stage2_valid,
            statistic="mean",
        ),
        "event_type_median": event_stat_adjustment(
            stage2_train,
            stage2_valid,
            statistic="median",
        ),
    }

    for model_spec in STAGE2_ML_SPECS:
        residual_model = model_spec["factory"]()
        residual_model.fit(
            stage2_train[STAGE2_FEATURES],
            stage2_train["event_residual"],
        )
        candidate_adjustments[model_spec["후보"]] = residual_model.predict(
            stage2_valid[STAGE2_FEATURES]
        )

    for candidate_name, adjustment in candidate_adjustments.items():
        adjustment = np.asarray(adjustment, dtype=float)
        final_prediction_raw = (
            stage2_valid["stage1_baseline_prediction"].to_numpy()
            + adjustment
        )
        final_prediction = np.clip(
            final_prediction_raw,
            a_min=0,
            a_max=None,
        )

        assert np.isfinite(adjustment).all()
        assert np.isfinite(final_prediction).all()

        stage2_result_rows.append(
            {
                "Fold": fold_spec["Fold"],
                "후보": candidate_name,
                "학습 이벤트 행": len(stage2_train),
                "검증 이벤트 행": len(stage2_valid),
                "평균 조정값": adjustment.mean(),
                "음수 최종 원예측": int((final_prediction_raw < 0).sum()),
                **regression_metrics(
                    stage2_valid[TARGET_COLUMN],
                    final_prediction,
                ),
            }
        )

stage2_fold_results = pd.DataFrame(stage2_result_rows)
stage2_summary = (
    stage2_fold_results.groupby("후보", as_index=False)
    .agg(
        Fold수=("Fold", "nunique"),
        평균_MAE=("MAE", "mean"),
        평균_RMSE=("RMSE", "mean"),
        평균_MAPE=("MAPE(%)", "mean"),
        평균_WAPE=("WAPE(%)", "mean"),
        MAE_표준편차=("MAE", "std"),
        음수최종원예측_합계=("음수 최종 원예측", "sum"),
    )
    .sort_values(["평균_MAE", "평균_RMSE"])
    .reset_index(drop=True)
)

assert TARGET_COLUMN not in STAGE2_FEATURES
assert set(STAGE2_FEATURES).issubset(stage2_event_oof.columns)
assert stage2_fold_results["후보"].nunique() == 10
assert stage2_summary["Fold수"].eq(2).all()

stage2_fold_definition = pd.DataFrame(
    [
        {
            "Fold": fold_spec["Fold"],
            "학습 OOF Fold": ", ".join(fold_spec["학습 OOF Fold"]),
            "검증 OOF Fold": fold_spec["검증 OOF Fold"],
        }
        for fold_spec in STAGE2_FOLD_SPECS
    ]
)

display(stage2_fold_definition)
display(pd.DataFrame({"Stage 2 ML 피처": STAGE2_FEATURES}))
display(stage2_fold_results)
display(stage2_summary)

,Fold,학습 OOF Fold,검증 OOF Fold
0,Stage 2 Fold 1,Fold 1,Fold 2
1,Stage 2 Fold 2,"Fold 1, Fold 2",Fold 3


,Stage 2 ML 피처
0,제1기분 자동차세
1,재산세(건축)
2,정기분 주민세
3,주민세(사업소분)
4,재산세(토지)
5,제2기분 자동차세
6,사회보험료 통합
7,event_count
8,month
9,day


,Fold,후보,학습 이벤트 행,검증 이벤트 행,평균 조정값,음수 최종 원예측,MAE,RMSE,MAPE(%),WAPE(%)
0,Stage 2 Fold 1,stage1_only,44,29,0.000,0,"52,123.640","101,610.940","4,710.478",56.779
1,Stage 2 Fold 1,global_mean,44,29,"37,432.508",0,"72,777.138","100,950.853","6,745.546",79.277
2,Stage 2 Fold 1,global_median,44,29,"-1,330.494",0,"51,791.645","101,887.730","4,638.675",56.417
3,Stage 2 Fold 1,event_type_mean,44,29,"7,069.190",0,"52,768.259","98,837.945","4,753.404",57.481
4,Stage 2 Fold 1,event_type_median,44,29,"-11,199.650",0,"49,293.496","103,752.683","3,996.289",53.696
5,Stage 2 Fold 1,ridge_alpha_1,44,29,"11,363.547",3,"75,598.805","118,527.074","6,057.585",82.351
6,Stage 2 Fold 1,ridge_alpha_10,44,29,"19,773.084",0,"64,733.903","104,811.210","5,764.096",70.515
7,Stage 2 Fold 1,random_forest,44,29,"6,345.682",0,"47,760.567","97,645.785","4,354.192",52.026
8,Stage 2 Fold 1,extra_trees,44,29,"12,481.535",0,"51,647.395","98,712.477","5,138.955",56.260
9,Stage 2 Fold 1,hist_gradient_boosting,44,29,"27,128.721",0,"60,504.174","103,660.757","5,084.503",65.908


,후보,Fold수,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE,MAE_표준편차,음수최종원예측_합계
0,event_type_median,2,"48,774.855","97,697.817","2,021.576",51.599,733.468,0
1,random_forest,2,"49,223.850","85,399.937","2,212.458",52.011,"2,069.394",0
2,global_median,2,"52,581.782","100,389.954","2,344.634",55.584,"1,117.423",0
3,stage1_only,2,"53,043.895","99,232.973","2,383.861",56.068,"1,301.437",0
4,event_type_mean,2,"53,832.316","90,836.849","2,414.911",56.897,"1,504.804",0
5,extra_trees,2,"54,496.983","88,107.889","2,610.923",57.544,"4,029.926",0
6,hist_gradient_boosting,2,"60,425.406","96,186.817","2,582.213",63.906,111.394,1
7,ridge_alpha_10,2,"61,455.808","94,319.805","2,924.440",65.098,"4,635.927",0
8,ridge_alpha_1,2,"68,166.843","102,671.193","3,074.083",72.327,"10,510.382",3
9,global_mean,2,"68,388.675","97,077.600","3,421.887",72.465,"6,206.224",0


## 13. Stage 2 후보 선택

두 시간순 검증 Fold의 평균 MAE를 1순위, 평균 RMSE를 2순위로 사용한다. `stage1_only`도 같은 조건에서 비교하여, 이벤트 조정 모델이 실제로 Stage 1 단독보다 나아지는지 확인한다.

In [13]:
selected_stage2_candidate = stage2_summary.iloc[0].to_dict()
stage1_only_stage2_result = stage2_summary.loc[
    stage2_summary["후보"].eq("stage1_only")
].iloc[0]

stage2_mae_improvement_vs_stage1_only = (
    (
        stage1_only_stage2_result["평균_MAE"]
        - selected_stage2_candidate["평균_MAE"]
    )
    / stage1_only_stage2_result["평균_MAE"]
    * 100
)

stage2_selection_comparison = pd.DataFrame(
    [
        {
            "구분": "Stage 1 단독",
            "후보": "stage1_only",
            "평균_MAE": stage1_only_stage2_result["평균_MAE"],
            "평균_RMSE": stage1_only_stage2_result["평균_RMSE"],
            "평균_MAPE": stage1_only_stage2_result["평균_MAPE"],
            "평균_WAPE": stage1_only_stage2_result["평균_WAPE"],
        },
        {
            "구분": "선택 후보",
            "후보": selected_stage2_candidate["후보"],
            "평균_MAE": selected_stage2_candidate["평균_MAE"],
            "평균_RMSE": selected_stage2_candidate["평균_RMSE"],
            "평균_MAPE": selected_stage2_candidate["평균_MAPE"],
            "평균_WAPE": selected_stage2_candidate["평균_WAPE"],
        },
    ]
)

selected_candidate_name = selected_stage2_candidate["후보"]
stage2_foldwise_comparison = (
    stage2_fold_results.loc[
        stage2_fold_results["후보"].isin(
            ["stage1_only", selected_candidate_name]
        ),
        ["Fold", "후보", "MAE", "RMSE", "WAPE(%)"],
    ]
    .pivot(index="Fold", columns="후보")
)
stage2_foldwise_comparison.columns = [
    f"{metric}_{candidate}"
    for metric, candidate in stage2_foldwise_comparison.columns
]
stage2_foldwise_comparison = stage2_foldwise_comparison.reset_index()
stage2_foldwise_comparison["MAE_개선율(%)"] = (
    (
        stage2_foldwise_comparison["MAE_stage1_only"]
        - stage2_foldwise_comparison[
            f"MAE_{selected_candidate_name}"
        ]
    )
    / stage2_foldwise_comparison["MAE_stage1_only"]
    * 100
)

lowest_volume_event_rows = stage2_event_oof.nsmallest(
    5,
    TARGET_COLUMN,
)[["접수일자", TARGET_COLUMN, "Fold", *EVENT_COLUMNS]]
mape_is_distorted = bool(
    lowest_volume_event_rows[TARGET_COLUMN].min() < 1_000
)

display(stage2_selection_comparison)
display(stage2_foldwise_comparison)
display(lowest_volume_event_rows)

print("Stage 2 이벤트 잔차 모델 비교 완료")
print(
    "- Stage 1 OOF 이벤트 행: "
    f"{len(stage2_event_oof):,}행"
)
print(
    "- Stage 2 시간순 검증 Fold: "
    f"{len(STAGE2_FOLD_SPECS)}개"
)
print(
    "- 선택 후보: "
    f"{selected_stage2_candidate['후보']} "
    f"(평균 MAE {selected_stage2_candidate['평균_MAE']:,.1f})"
)
print(
    "- Stage 1 단독 대비 MAE 개선율: "
    f"{stage2_mae_improvement_vs_stage1_only:,.2f}%"
)
print(
    "- 선택 후보가 MAE를 개선한 검증 Fold: "
    f"{int(stage2_foldwise_comparison['MAE_개선율(%)'].gt(0).sum())}/"
    f"{len(stage2_foldwise_comparison)}개"
)
if mape_is_distorted:
    print(
        "- 주의: 실제 물량이 1,000통 미만인 이벤트 행 때문에 "
        "MAPE가 과도하게 커져 모델 선정에는 MAE와 RMSE를 사용함"
    )
print("- 2026년 Test 성능 확인: 수행하지 않음")

,구분,후보,평균_MAE,평균_RMSE,평균_MAPE,평균_WAPE
0,Stage 1 단독,stage1_only,"53,043.895","99,232.973","2,383.861",56.068
1,선택 후보,event_type_median,"48,774.855","97,697.817","2,021.576",51.599


,Fold,MAE_event_type_median,MAE_stage1_only,RMSE_event_type_median,RMSE_stage1_only,WAPE(%)_event_type_median,WAPE(%)_stage1_only,MAE_개선율(%)
0,Stage 2 Fold 1,"49,293.496","52,123.640","103,752.683","101,610.940",53.696,56.779,5.430
1,Stage 2 Fold 2,"48,256.215","53,964.150","91,642.950","96,855.007",49.502,55.358,10.577


,접수일자,접수통수,Fold,제1기분 자동차세,재산세(건축),정기분 주민세,주민세(사업소분),재산세(토지),제2기분 자동차세,사회보험료 통합
240,2025-06-22,65,Fold 2,0,0,0,0,0,0,1
262,2025-07-22,20168,Fold 3,0,0,0,0,0,0,1
264,2025-07-24,20714,Fold 3,0,0,0,0,0,0,1
242,2025-06-24,21190,Fold 2,0,0,0,0,0,0,1
268,2025-07-30,21876,Fold 3,0,0,0,1,0,0,0


Stage 2 이벤트 잔차 모델 비교 완료
- Stage 1 OOF 이벤트 행: 119행
- Stage 2 시간순 검증 Fold: 2개
- 선택 후보: event_type_median (평균 MAE 48,774.9)
- Stage 1 단독 대비 MAE 개선율: 8.05%
- 선택 후보가 MAE를 개선한 검증 Fold: 2/2개
- 주의: 실제 물량이 1,000통 미만인 이벤트 행 때문에 MAPE가 과도하게 커져 모델 선정에는 MAE와 RMSE를 사용함
- 2026년 Test 성능 확인: 수행하지 않음
